# 0824_lsw_004_imbalance_handling

검사유형별 5분리 구조(`0824_lsw_003_structure_comparison`에서 채택) 위에서 클래스 불균형 대응 기법을 비교합니다. 데이터 전처리와 평가 함수는 003과 동일하게 재사용합니다.

이번 노트북은 두 부분으로 구성됩니다: (1) SMOTE/ADASYN 적용 전에 소수 클래스(불량)에 이상치가 얼마나 있는지 가볍게 확인, (2) 불균형 처리 기법별 비교.

## 1. 설정과 라이브러리

In [1]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, confusion_matrix, roc_auc_score
from xgboost import XGBClassifier

EXPERIMENT_ID = "0824_lsw_004_imbalance_handling"
RANDOM_STATE = 42
DATA_PATH = Path("../data/raw/dataset.csv")
TARGET = "class"
TIME_COLUMN = "timestamp"
RECORD_ID = "record_id"
MODEL_DIR = Path("../models")

COST_SCENARIOS = {"1:10": (1, 10), "1:100": (1, 100)}

assert DATA_PATH.exists(), f"파일을 찾을 수 없습니다: {DATA_PATH.resolve()}"
print("experiment:", EXPERIMENT_ID)

experiment: 0824_lsw_004_imbalance_handling


## 2. 데이터 로딩·전처리 (003과 동일)

In [2]:
raw_df = pd.read_csv(DATA_PATH, low_memory=False)
source_index_column = raw_df.columns[0]
if source_index_column.startswith("Unnamed:"):
    raw_df = raw_df.rename(columns={source_index_column: RECORD_ID})
elif source_index_column != RECORD_ID:
    raise ValueError(f"예상하지 못한 첫 번째 컬럼: {source_index_column}")
assert raw_df[RECORD_ID].is_unique, "record_id가 고유하지 않습니다."

dedup_columns = [c for c in raw_df.columns if c not in {RECORD_ID, TIME_COLUMN}]
duplicate_mask = raw_df.duplicated(subset=dedup_columns, keep="first")
clean_df = raw_df.loc[~duplicate_mask].copy().reset_index(drop=True)

clean_df[TIME_COLUMN] = pd.to_datetime(clean_df[TIME_COLUMN], errors="raise", utc=True)
clean_df = clean_df.sort_values([TIME_COLUMN, RECORD_ID], kind="stable").reset_index(drop=True)

feature_columns_all = [c for c in clean_df.columns if c not in {RECORD_ID, TIME_COLUMN, TARGET}]

timestamps = clean_df[TIME_COLUMN]
timestamp_group_sizes = timestamps.value_counts(sort=False).sort_index()
cumulative_rows = timestamp_group_sizes.cumsum().to_numpy()
train_end_time = timestamp_group_sizes.index[int(np.searchsorted(cumulative_rows, len(clean_df) * 0.60, side="left"))]
valid_end_time = timestamp_group_sizes.index[int(np.searchsorted(cumulative_rows, len(clean_df) * 0.80, side="left"))]

train_mask = timestamps <= train_end_time
valid_mask = (timestamps > train_end_time) & (timestamps <= valid_end_time)
test_mask = timestamps > valid_end_time

print("rows_after_dedup:", len(clean_df))
pd.DataFrame(
    [
        {"split": name, "rows": int(mask.sum())}
        for name, mask in [("train", train_mask), ("validation", valid_mask), ("test", test_mask)]
    ]
).set_index("split")

rows_after_dedup: 391992


,rows
split,
train,235222
validation,78374
test,78396


## 3. 평가 함수 (003과 동일)

In [3]:
def slip_rate(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    actual_positive = y_true == 1
    if actual_positive.sum() == 0:
        return 0.0
    fn = ((y_pred == 0) & actual_positive).sum()
    return fn / actual_positive.sum()


def volume_reduction(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    actual_negative = y_true == 0
    if actual_negative.sum() == 0:
        return 0.0
    tn = ((y_pred == 0) & actual_negative).sum()
    return tn / actual_negative.sum()


def total_cost(y_true, y_pred, cost_fp, cost_fn):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    fn = ((y_pred == 0) & (y_true == 1)).sum()
    fp = ((y_pred == 1) & (y_true == 0)).sum()
    return fn * cost_fn + fp * cost_fp


def select_threshold(y_val, proba_val, max_slip_rate=0.01):
    candidates = np.sort(np.unique(proba_val))[::-1]
    for t in candidates:
        y_pred = (proba_val >= t).astype(int)
        if slip_rate(y_val, y_pred) <= max_slip_rate:
            return float(t)
    return 0.0


def evaluate_at_threshold(y_true, proba, threshold):
    y_pred = (proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    result = {
        "threshold": threshold,
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
        "slip_rate": slip_rate(y_true, y_pred),
        "volume_reduction": volume_reduction(y_true, y_pred),
        "pr_auc": average_precision_score(y_true, proba),
        "roc_auc": roc_auc_score(y_true, proba) if len(np.unique(y_true)) > 1 else float("nan"),
    }
    for name, (cost_fp, cost_fn) in COST_SCENARIOS.items():
        result[f"total_cost_{name}"] = total_cost(y_true, y_pred, cost_fp, cost_fn)
    return result


def get_non_constant_columns(candidate_columns, train_frame):
    nunique = train_frame[candidate_columns].nunique()
    return nunique[nunique > 1].index.tolist()

## 4. 검사유형별 subset 준비

`0824_lsw_003`에서 채택한 구조(검사유형별 5분리, 유형별 임계값)를 이번 실험의 baseline으로 그대로 재현합니다.

In [4]:
type_splits = {}
for inspection_type in sorted(clean_df["inspection_type"].unique()):
    type_mask = clean_df["inspection_type"] == inspection_type
    type_train_df = clean_df.loc[train_mask & type_mask]
    type_valid_df = clean_df.loc[valid_mask & type_mask]
    type_test_df = clean_df.loc[test_mask & type_mask]
    type_feature_columns = get_non_constant_columns(feature_columns_all, type_train_df)
    type_splits[inspection_type] = {
        "train": type_train_df,
        "valid": type_valid_df,
        "test": type_test_df,
        "feature_columns": type_feature_columns,
    }
    print(f"type {inspection_type}: train={len(type_train_df)}, features={len(type_feature_columns)}, "
          f"train_pos={int((type_train_df[TARGET]==1).sum())}")

type 0: train=41961, features=47, train_pos=57
type 1: train=34041, features=34, train_pos=555


type 2: train=74910, features=24, train_pos=579


type 3: train=80792, features=22, train_pos=620
type 4: train=3518, features=24, train_pos=13


## 5. SMOTE/ADASYN 적용 전 — 소수 클래스(불량) 이상치 체크

SMOTE/ADASYN은 최근접 이웃 사이를 보간해 합성 샘플을 만들기 때문에, 소수 클래스(`class=1`, 실제 불량)에 이상치가 있으면 그 이상치와 정상적인 불량 샘플 사이를 보간해 이상한 합성 샘플을 만들 위험이 있습니다.

방법: 각 검사유형의 Train 전체(정상+불량)로 피처별 IQR 범위(`Q1-1.5·IQR` ~ `Q3+1.5·IQR`, "정상 운영 범위")를 구하고, 불량(`class=1`) 샘플 각각이 이 범위를 벗어나는 피처 비율을 계산합니다. 피처의 20% 이상이 범위를 벗어나면 "이상치 후보"로 표시합니다.

In [5]:
OUTLIER_FEATURE_FRACTION_THRESHOLD = 0.2

outlier_summary_rows = []
minority_outlier_flags = {}

for inspection_type, split in type_splits.items():
    train_df = split["train"]
    feature_columns = split["feature_columns"]

    q1 = train_df[feature_columns].quantile(0.25)
    q3 = train_df[feature_columns].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    minority_df = train_df.loc[train_df[TARGET] == 1, feature_columns]
    is_outside = (minority_df < lower) | (minority_df > upper)
    outlier_feature_fraction = is_outside.mean(axis=1)
    is_flagged = outlier_feature_fraction > OUTLIER_FEATURE_FRACTION_THRESHOLD

    minority_outlier_flags[inspection_type] = outlier_feature_fraction

    outlier_summary_rows.append(
        {
            "inspection_type": inspection_type,
            "n_minority_train": len(minority_df),
            "n_flagged": int(is_flagged.sum()),
            "flagged_pct": is_flagged.mean() if len(minority_df) else float("nan"),
            "median_outlier_feature_fraction": outlier_feature_fraction.median() if len(minority_df) else float("nan"),
        }
    )

outlier_summary_df = pd.DataFrame(outlier_summary_rows).set_index("inspection_type")
outlier_summary_df

,n_minority_train,n_flagged,flagged_pct,median_outlier_feature_fraction
inspection_type,,,,
0,57,28,0.491228,0.191489
1,555,236,0.425225,0.176471
2,579,36,0.062176,0.041667
3,620,52,0.083871,0.090909
4,13,7,0.538462,0.208333


## 6. 결론 및 다음 단계 (1차 — 이상치 체크까지)

### 결과

| inspection_type | Train 불량 수 | 이상치 후보 | 비율 | 중위 이상치-피처 비율 |
|---|---:|---:|---:|---:|
| 0 | 57 | 28 | 49.1% | 19.1% |
| 1 | 555 | 236 | 42.5% | 17.6% |
| 2 | 579 | 36 | 6.2% | 4.2% |
| 3 | 620 | 52 | 8.4% | 9.1% |
| 4 | 13 | 7 | 53.8% | 20.8% |

### 핵심 관찰

- **type0/1/4는 불량 샘플의 40~54%가 이상치 후보다.** 특히 type0(49.1%)와 type4(53.8%, 표본 13개뿐이라 노이즈 큼)는 절반 가까이가 "정상 운영 범위를 크게 벗어난" 불량이다. type1도 555건 중 236건(42.5%)으로 적지 않다.
- **type2/3는 상대적으로 이상치가 적다**(6.2%, 8.4%) — 이 두 유형은 SMOTE/ADASYN을 비교적 안전하게 적용할 수 있을 것으로 보인다.
- type0/1/4처럼 이상치 비율이 높은 유형에서 SMOTE를 그대로 적용하면, 이상치와 정상 불량 사이를 보간해 실제로는 없는 패턴의 합성 샘플을 대량 생성할 위험이 있다. 이는 003에서 발견한 "일부 불량 샘플에 극단적으로 낮은 예측확률이 나온다"는 현상과도 연결되는 관찰이다 — 이 이상치들이 바로 그 "애매한 불량"일 가능성이 있다.

### 다음 단계

- type2/3는 SMOTE/ADASYN을 바로 시도해도 된다.
- type0/1/4는 이상치를 제거·완화한 버전과 그대로 둔 버전을 나란히 비교하는 시도 로그를 추가한다 (이상치 제거가 실제로 도움되는지는 직접 비교해야 알 수 있음 — 제거가 항상 정답은 아니고, 그 이상치가 진짜 어려운 불량 패턴일 수도 있음).
- 다음 셀부터 불균형 처리 기법(class_weight/scale_pos_weight, SMOTE, ADASYN, undersampling)을 유형별로 비교한다.
